<a href="https://colab.research.google.com/github/greidias-ctrl/Cienciadedados/blob/main/Grupo1_CienciaDeDados_aplicada_ao_MKT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Carregando os dados

In [1]:
from google.colab import files
files.upload()

Saving campanhas_marketing_digital.csv to campanhas_marketing_digital.csv


{'campanhas_marketing_digital.csv': b'\xef\xbb\xbfdata,canal,campanha,etapa_funil,investimento,impressoes,alcance,cliques,interacoes,leads,conversoes,receita\r\n2026-08-03,Google Ads,Pesquisa Marca,Fundo de funil,1200,52000,41000,2600,310,150,58,8700\r\n2026-08-03,Meta Ads,Conversao Social,Meio de funil,1000,78000,59000,2340,410,118,42,5200\r\n2026-08-03,Instagram,Conteudo Impulsionado,Topo de funil,650,92000,68000,1840,520,72,18,1800\r\n2026-08-03,LinkedIn Ads,Leads B2B,Meio de funil,900,31000,24500,930,145,64,16,4800\r\n2026-08-10,Google Ads,Pesquisa Marca,Fundo de funil,1250,54000,42500,2862,335,162,63,9450\r\n2026-08-10,Meta Ads,Conversao Social,Meio de funil,1050,81000,61200,2511,438,126,46,5750\r\n2026-08-10,Instagram ,Conteudo Impulsionado,Topo de funil,700,96000,70400,2016,548,78,20,2000\r\n2026-08-10,LinkedIn Ads,Leads B2B,Meio de funil,950,32500,25200,975,151,68,17,5100\r\n2026-08-17,Google Ads,Pesquisa Marca,Fundo de funil,1300,56000,43800,3024,352,171,67,10050\r\n2026-08-17

Mostrando Planilha


In [2]:
import pandas as pd

df = pd.read_csv('campanhas_marketing_digital - campanhas_marketing_digital.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'campanhas_marketing_digital - campanhas_marketing_digital.csv'

Diagnóstico da base

In [ ]:
print('Linhas e colunas:', df.shape)
df.info()

Resposta do diagnostico limpeza de dados

In [ ]:
print('Valores ausentes por coluna:')
print(df.isna().sum())
print('\nLinhas duplicadas:', df.duplicated().sum())

Qual linha tem o dado NaN

In [ ]:
df[df['leads'].isna()]

Verificação de consistência


In [ ]:
df['canal'].map(repr).value_counts()

Processamento

In [ ]:
df['canal'] = df['canal'].str.strip()
df['data'] = pd.to_datetime(df['data'])

df.info()

In [ ]:
print(df['canal'].value_counts())
print('\nPeríodo analisado:', df['data'].min().date(), 'até', df['data'].max().date())
print('Semanas distintas:', df['data'].nunique())

Análise exploratória

In [ ]:
def calcular_metricas(tabela):
    tabela = tabela.copy()
    tabela['CTR_%'] = tabela['cliques'] / tabela['impressoes'] * 100
    tabela['tx_conversao_%'] = tabela['conversoes'] / tabela['cliques'] * 100
    tabela['CPC'] = tabela['investimento'] / tabela['cliques']
    tabela['CPA'] = tabela['investimento'] / tabela['conversoes']
    tabela['ROAS'] = tabela['receita'] / tabela['investimento']
    return tabela

df = calcular_metricas(df)
df[['data', 'canal', 'CTR_%', 'tx_conversao_%', 'CPC', 'CPA', 'ROAS']].round(2).head(8)

Célula em um único registro

In [ ]:
registro = df[(df['canal'] == 'Google Ads') & (df['data'] == '2026-08-03')]
registro[['CTR_%', 'tx_conversao_%', 'CPC', 'CPA', 'ROAS']].round(2)

Soma dos valores totais por plataforma


In [ ]:
colunas_soma = ['investimento', 'impressoes', 'alcance', 'cliques', 'interacoes', 'leads', 'conversoes', 'receita']

por_canal = df.groupby('canal')[colunas_soma].sum()
por_canal = calcular_metricas(por_canal)

por_canal['part_investimento_%'] = por_canal['investimento'] / por_canal['investimento'].sum() * 100
por_canal['part_receita_%'] = por_canal['receita'] / por_canal['receita'].sum() * 100

por_canal[['investimento', 'conversoes', 'receita', 'CTR_%', 'tx_conversao_%', 'CPC', 'CPA', 'ROAS',
           'part_investimento_%', 'part_receita_%']].sort_values('ROAS', ascending=False).round(2)

Maior de todos foi o Google Search

In [ ]:
print('Maior volume de conversões:', por_canal['conversoes'].idxmax())
print('Maior taxa de conversão:', por_canal['tx_conversao_%'].idxmax())
print('Maior ROAS:', por_canal['ROAS'].idxmax())
print('Menor CPA:', por_canal['CPA'].idxmin())

Importando bibliotecas gráficas


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
cores = {'Google Ads': '#4C72B0', 'Meta Ads': '#DD8452', 'Instagram': '#C44E9C', 'LinkedIn Ads': '#55A868'}

Conversões por plataforma


In [ ]:
ordem = por_canal.sort_values('conversoes', ascending=False).index

plt.figure(figsize=(8, 4.5))
sns.barplot(x=ordem, y=por_canal.loc[ordem, 'conversoes'], hue=ordem, palette=cores, legend=False)
plt.title('Gráfico 1 — Total de conversões por canal (03/08 a 07/09/2026)')
plt.xlabel('')
plt.ylabel('Conversões')
plt.show()

Conversões ao longo do tempo

In [ ]:
plt.figure(figsize=(9, 4.5))
sns.lineplot(data=df, x='data', y='conversoes', hue='canal', palette=cores, marker='o')
plt.title('Gráfico 2 — Conversões semanais por canal')
plt.xlabel('Semana')
plt.ylabel('Conversões')
plt.legend(title='Canal')
plt.show()

Disperção entre investimento e conversões

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='investimento', y='conversoes', hue='canal', palette=cores, s=80)
plt.title('Gráfico 3 — Investimento x conversões (cada ponto = 1 semana de 1 canal)')
plt.xlabel('Investimento semanal (R$)')
plt.ylabel('Conversões')
plt.show()

print('Correlação entre investimento e conversões:', round(df['investimento'].corr(df['conversoes']), 3))

Insights e recomendações

In [ ]:
por_canal['ticket_medio'] = por_canal['receita'] / por_canal['conversoes']
por_canal[['conversoes', 'receita', 'ticket_medio', 'CPA', 'ROAS']].sort_values('ticket_medio', ascending=False).round(2)

Três insights

1. Resultado positivo - o Google Ads escala com eficiência
Evidência: o Google Ads recebeu 30,9% do investimento e gerou 42,9% da receita, com ROAS de 7,56 e o menor CPA da base (R$ 19,83). Ao longo das seis semanas, o investimento cresceu 33,3% e as conversões, 37,9%.
Interpretação: o canal manteve a eficiência mesmo recebendo mais verba, dentro da faixa observada.

2. Resultado que merece atenção — retorno decrescente no Instagram
Evidência: o investimento no Instagram cresceu 53,8% e as conversões, apenas 33,3%. O canal consome 18,1% da verba e devolve 8,8% da receita (ROAS de 2,65).
Interpretação: cada real adicional está trazendo menos resultado. Porém, por ser um canal de topo de funil, parte do seu valor pode não aparecer em conversões diretas.

3. Oportunidade — o LinkedIn tem clientes valiosos, mas caros de converter**
Evidência: o LinkedIn tem o CPA mais alto (R$ 58,56), mas cada conversão vale em média R$ 300, o dobro do Google (R$ 150). Por isso seu ROAS (5,12) é praticamente igual ao do Meta (5,20).
Interpretação: o problema do LinkedIn não é o valor do cliente, e sim o custo para convertê-lo. Sua taxa de conversão (1,82%) tem espaço para melhorar.

Explicação:
A campanha do Google Ads é de pesquisa de marca: ela alcança pessoas que já estão procurando a empresa, ou seja, com alta intenção de compra. Parte dessa demanda pode ter sido criada pelo Instagram e pelo Meta, que trabalham no topo e no meio do funil. Se a receita for atribuída ao último clique, o Google pode estar recebendo crédito por vendas que outros canais ajudaram a gerar. Esta é uma hipótese, não uma conclusão: a base não tem dados de atribuição para confirmá-la.

Recomendações

Recomendação 1 — Ampliar o Google Ads de forma gradual e monitorada
- Evidência: maior ROAS (7,56), menor CPA (R\$ 19,83) e eficiência mantida ao escalar (+33,3% de investimento, +37,9% de conversões).
- Decisão proposta: aumentar o investimento em 15% a 20% nas próximas 4 semanas.
- Validação: acompanhar o CPA semanalmente. Se ultrapassar R\$ 24 (patamar do Meta Ads), o aumento deve ser revisto. Campanhas de pesquisa de marca têm um teto, porque dependem de quantas pessoas buscam a marca.

Recomendação 2 — Testar o Instagram antes de cortar ou ampliar
- Evidência: menor ROAS (2,65) e retorno decrescente (+53,8% de investimento para +33,3% de conversões), mas com papel de topo de funil.
- Decisão proposta: manter o investimento do Instagram no nível atual e realizar um teste de incrementalidade, por exemplo pausando o canal em uma região por 2 a 4 semanas.
- Validação: comparar as buscas pela marca e as conversões no Google entre a região com e sem Instagram. Se caírem sem o Instagram, o canal está alimentando o fundo do funil e deve ser mantido.

Conclusão

A análise mostra que o Google Ads concentra volume e eficiência: com 31% do investimento, gerou 43% da receita, com ROAS de 7,56 e o menor CPA da base. O Meta Ads ocupa uma posição intermediária consistente. O LinkedIn tem conversões caras, mas ROAS competitivo, graças ao ticket médio mais alto do público B2B. O Instagram apresenta o menor retorno direto, porém atua no topo do funil e não deve ser avaliado apenas por conversões. Como investimento e conversões cresceram juntos, não é possível afirmar causalidade sem experimento. Recomendamos ampliar o Google Ads de forma gradual e testar a contribuição do Instagram antes de qualquer corte. As conclusões são limitadas por uma base curta, fictícia e sem dados de margem ou atribuição.

Registro das limitações

Base de dados de um período curto: 24 registros, 6 semanas e uma única campanha por canal. Os padrões podem não se repetir em períodos mais longos.

Dado ausente: o lead do Instagram em 24/08 foi mantido como vazio, então o total de leads desse canal está subestimado.

ROAS não é lucro nem ROI: a base não inclui margem, custos operacionais ou impostos. Um ROAS alto não garante que a campanha seja lucrativa.

Sem modelo de atribuição: a receita foi atribuída a um único canal, mas o cliente pode ter passado por vários antes de comprar. O Instagram pode estar gerando demanda que converte no Google.

Correlação não é causalidade: a associação de 0,79 entre investimento e conversões não prova que mais verba causa mais vendas.

Dados que poderiam complementar a analise : novos clientes e recorrentes, LTV, público, criativo, dispositivo e sazonalidade.
